# Stage 08C-3 — Public Baseline/SOTA Evaluation for HMRNet Revision

Notebook Google Colab ini menjalankan **ProdLDA, CombinedTM, dan BERTopic** pada **StackOverflow** dan **20 Newsgroups**, menggunakan feature bundle Stage 08B yang sudah dibuat. Hasilnya dihitung dengan 10 random seeds dan metrik yang sama dengan Stage 08C-1/2R: **Cv, C_NPMI, UMass, Topic Diversity, Topic Redundancy, Topic Stability, NMI, dan ARI**.

Pada akhir notebook, hasil otomatis digabung dengan **LDA/NMF** dari Stage 08C-1 dan **HMRNet-Final** dari Stage 08C-2R jika file hasilnya tersedia.

**Disarankan: Google Colab GPU.**


In [ ]:
# Install dependencies
!pip -q install -U gensim contextualized-topic-models bertopic umap-learn hdbscan


In [ ]:
import os, gc, json, math, time, random, warnings, itertools, platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import sparse
from scipy.optimize import linear_sum_assignment
from scipy.stats import entropy
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
warnings.filterwarnings("ignore")

BASE_DIR="/content/HMRNet_Public_Datasets"
FEATURE_DIR=os.path.join(BASE_DIR,"stage08b_features")
STAGE08C1_DIR=os.path.join(BASE_DIR,"stage08c1_results")
STAGE08C2R_DIR=os.path.join(BASE_DIR,"stage08c2r_results")
OUTPUT_DIR=os.path.join(BASE_DIR,"stage08c3_results")
os.makedirs(OUTPUT_DIR,exist_ok=True)

DATASETS=["StackOverflow","20Newsgroups"]
SEEDS=list(range(1,11))
K=20
TOP_N=10
BATCH_SIZE=128
LEARNING_RATE=0.002
MAX_EPOCHS=60
PATIENCE=10
DROPOUT=0.20
VALIDATION_FRACTION=0.10
CTM_HIDDEN=(100,100)
CTM_N_SAMPLES_THETA=10
BERTOPIC_MIN_CLUSTER_SIZE=15
BERTOPIC_N_NEIGHBORS=15
BERTOPIC_UMAP_COMPONENTS=5
DEVICE="cuda" if torch.cuda.is_available() else "cpu"

packages=["torch","numpy","pandas","scikit-learn","gensim","contextualized-topic-models","bertopic","umap-learn","hdbscan"]
versions={}
for p in packages:
    try: versions[p]=metadata.version(p)
    except Exception: versions[p]="not detected"

print("="*90)
print("STAGE 08C-3 — ProdLDA + CombinedTM + BERTopic")
print("="*90)
print("Device:",DEVICE)
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))
print("Datasets:",DATASETS)
print("Seeds:",SEEDS)
print("K:",K)


In [ ]:
def set_seed(seed:int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic=True
        torch.backends.cudnn.benchmark=False
    except Exception: pass

def load_dataset(name):
    d=os.path.join(FEATURE_DIR,name)
    req=["bow_counts.npz","semantic_embeddings.npy","labels.npy","documents.csv","vocabulary.csv"]
    for f in req:
        p=os.path.join(d,f)
        if not os.path.exists(p): raise FileNotFoundError(p)
    bow=sparse.load_npz(os.path.join(d,"bow_counts.npz")).tocsr()
    sem=np.load(os.path.join(d,"semantic_embeddings.npy"),mmap_mode="r")
    labels=np.load(os.path.join(d,"labels.npy"),allow_pickle=True)
    docs=pd.read_csv(os.path.join(d,"documents.csv"))
    vdf=pd.read_csv(os.path.join(d,"vocabulary.csv"))
    vocab=vdf.sort_values("term_id")["term"].astype(str).to_numpy()
    assert bow.shape[0]==sem.shape[0]==len(labels)==len(docs)
    assert bow.shape[1]==len(vocab)
    return {"bow":bow,"semantic":sem,"labels":labels,"documents":docs,"vocabulary":vocab}

def coherence_inputs(docs,vocab):
    vs=set(vocab.tolist())
    texts=[[t for t in str(x).split() if t in vs] for x in docs["text_clean"].fillna("").astype(str)]
    dictionary=Dictionary(texts)
    corpus=[dictionary.doc2bow(t) for t in texts]
    return texts,dictionary,corpus

def normalize_beta(beta):
    beta=np.asarray(beta,dtype=np.float64)
    beta=np.maximum(beta,0.0)
    s=beta.sum(axis=1,keepdims=True); s[s==0]=1.0
    return beta/s

def topic_words(beta,vocab,top_n=10):
    return [[str(vocab[i]) for i in np.argsort(row)[::-1][:top_n]] for row in beta]

def topic_diversity(topics):
    words=[w for t in topics for w in t]
    return len(set(words))/len(words) if words else np.nan

def topic_redundancy(topics):
    vals=[]
    for i in range(len(topics)):
        a=set(topics[i])
        for j in range(i+1,len(topics)):
            b=set(topics[j]); u=a|b
            vals.append(len(a&b)/len(u) if u else 0.0)
    return float(np.mean(vals)) if vals else np.nan

def coherence_metrics(topics,texts,dictionary,corpus):
    cv=float(CoherenceModel(topics=topics,texts=texts,dictionary=dictionary,coherence="c_v",topn=TOP_N,processes=1).get_coherence())
    npmi=float(CoherenceModel(topics=topics,texts=texts,dictionary=dictionary,coherence="c_npmi",topn=TOP_N,processes=1).get_coherence())
    umass=float(CoherenceModel(topics=topics,corpus=corpus,dictionary=dictionary,coherence="u_mass",topn=TOP_N,processes=1).get_coherence())
    return cv,npmi,umass

def jsd(p,q):
    p=np.maximum(np.asarray(p,dtype=np.float64),1e-12); q=np.maximum(np.asarray(q,dtype=np.float64),1e-12)
    p/=p.sum(); q/=q.sum(); m=.5*(p+q)
    return float(.5*entropy(p,m,base=2)+.5*entropy(q,m,base=2))

def stability_pair(a,b):
    sim=np.zeros((a.shape[0],b.shape[0]),dtype=np.float64)
    for i in range(a.shape[0]):
        for j in range(b.shape[0]): sim[i,j]=1.0-jsd(a[i],b[j])
    r,c=linear_sum_assignment(-sim)
    return float(sim[r,c].mean())

def multiseed_stability(beta_runs,seeds):
    rec=[]; vals=[]
    for i,j in itertools.combinations(range(len(beta_runs)),2):
        s=stability_pair(beta_runs[i],beta_runs[j]); vals.append(s); rec.append({"seed_i":seeds[i],"seed_j":seeds[j],"stability":s})
    vals=np.asarray(vals,dtype=float)
    if not len(vals): return {"mean":np.nan,"std":np.nan,"ci_low":np.nan,"ci_high":np.nan,"n_pairs":0,"records":rec}
    mean=float(vals.mean()); std=float(vals.std(ddof=1)) if len(vals)>1 else 0.0; se=std/math.sqrt(len(vals))
    return {"mean":mean,"std":std,"ci_low":mean-1.96*se,"ci_high":mean+1.96*se,"n_pairs":len(vals),"records":rec}

def build_result(dataset,model,seed,beta,theta_or_assign,labels,vocab,texts,dictionary,corpus,runtime,assignments_are_labels=False):
    beta=normalize_beta(beta); topics=topic_words(beta,vocab,TOP_N)
    cv,npmi,umass=coherence_metrics(topics,texts,dictionary,corpus)
    td=topic_diversity(topics); tr=topic_redundancy(topics)
    dominant=np.asarray(theta_or_assign) if assignments_are_labels else np.argmax(np.asarray(theta_or_assign),axis=1)
    nmi=float(normalized_mutual_info_score(labels,dominant)); ari=float(adjusted_rand_score(labels,dominant))
    result={"dataset":dataset,"model":model,"seed":seed,"n_topics_observed":int(beta.shape[0]),"Cv":cv,"C_NPMI":npmi,"UMass":umass,"Topic_Diversity":td,"Topic_Redundancy":tr,"NMI":nmi,"ARI":ari,"runtime_seconds":float(runtime)}
    rows=[{"dataset":dataset,"model":model,"seed":seed,"topic_id":i,"top_words":", ".join(words)} for i,words in enumerate(topics)]
    return result,rows,beta

def show_result(r):
    print(f"Cv={r['Cv']:.4f} | CNPMI={r['C_NPMI']:.4f} | UMass={r['UMass']:.4f} | TD={r['Topic_Diversity']:.4f} | TR={r['Topic_Redundancy']:.4f} | NMI={r['NMI']:.4f} | ARI={r['ARI']:.4f} | topics={r['n_topics_observed']}")


In [ ]:
class SparseBowDataset(Dataset):
    def __init__(self,bow,indices): self.bow=bow; self.indices=np.asarray(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self,i): return torch.from_numpy(self.bow[self.indices[i]].toarray().ravel().astype(np.float32))

class ProdLDANet(nn.Module):
    def __init__(self,vocab_size,n_topics=20,hidden=100,dropout=.20):
        super().__init__()
        self.encoder=nn.Sequential(nn.Linear(vocab_size,hidden),nn.Softplus(),nn.Dropout(dropout),nn.Linear(hidden,hidden),nn.Softplus(),nn.Dropout(dropout))
        self.mu=nn.Linear(hidden,n_topics); self.logvar=nn.Linear(hidden,n_topics)
        self.beta_logits=nn.Parameter(torch.empty(n_topics,vocab_size)); nn.init.xavier_uniform_(self.beta_logits)
    def forward(self,x,sample=True):
        xn=x/x.sum(dim=1,keepdim=True).clamp_min(1.0); h=self.encoder(xn); mu=self.mu(h); logvar=self.logvar(h).clamp(-10,10)
        z=mu+torch.randn_like(mu)*torch.exp(.5*logvar) if sample else mu
        theta=torch.softmax(z,dim=1); log_prob=F.log_softmax(theta@self.beta_logits,dim=1)
        return mu,logvar,theta,log_prob
    def beta(self): return torch.softmax(self.beta_logits,dim=1)

def prodlda_loss(x,mu,logvar,log_prob):
    lengths=x.sum(dim=1).clamp_min(1.0)
    rec=(-(x*log_prob).sum(dim=1)/lengths).mean()
    kl=(-.5*(1+logvar-mu.pow(2)-logvar.exp()).sum(dim=1)).mean()
    return rec+kl

def fit_prodlda(bow,seed):
    set_seed(seed); n=bow.shape[0]; rng=np.random.default_rng(seed); idx=rng.permutation(n); nv=max(1,int(round(n*VALIDATION_FRACTION)))
    va=idx[:nv]; tr=idx[nv:]
    train_loader=DataLoader(SparseBowDataset(bow,tr),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
    val_loader=DataLoader(SparseBowDataset(bow,va),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    model=ProdLDANet(bow.shape[1],K,100,DROPOUT).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    best=float("inf"); state=None; bad=0
    for epoch in range(1,MAX_EPOCHS+1):
        model.train(); tsum=0.; tn=0
        for x in train_loader:
            x=x.to(DEVICE); opt.zero_grad(set_to_none=True); mu,lv,th,lp=model(x,True); loss=prodlda_loss(x,mu,lv,lp); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step(); tsum+=float(loss.detach().cpu())*x.shape[0]; tn+=x.shape[0]
        model.eval(); vsum=0.; vn=0
        with torch.no_grad():
            for x in val_loader:
                x=x.to(DEVICE); mu,lv,th,lp=model(x,False); loss=prodlda_loss(x,mu,lv,lp); vsum+=float(loss.cpu())*x.shape[0]; vn+=x.shape[0]
        vl=vsum/max(vn,1)
        if epoch==1 or epoch%10==0: print(f"  ProdLDA epoch {epoch:02d} | train={tsum/max(tn,1):.4f} | val={vl:.4f}")
        if vl<best-1e-5:
            best=vl; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=PATIENCE: break
    if state is not None: model.load_state_dict(state)
    loader=DataLoader(SparseBowDataset(bow,np.arange(n)),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    parts=[]; model.eval()
    with torch.no_grad():
        for x in loader:
            _,_,theta,_=model(x.to(DEVICE),False); parts.append(theta.cpu().numpy())
    return np.vstack(parts),model.beta().detach().cpu().numpy(),model

def run_prodlda(name,data,coh):
    bow=data['bow']; labels=data['labels']; vocab=data['vocabulary']; texts,dictionary,corpus=coh
    results=[]; rows=[]; betas=[]; ok=[]
    for seed in SEEDS:
        print("\n"+"-"*90); print(f"{name} | ProdLDA | seed={seed}"); print("-"*90); t=time.time()
        try:
            theta,beta,model=fit_prodlda(bow,seed)
            r,rr,beta=build_result(name,"ProdLDA",seed,beta,theta,labels,vocab,texts,dictionary,corpus,time.time()-t)
            results.append(r); rows.extend(rr); betas.append(beta); ok.append(seed); show_result(r); del theta,beta,model
        except Exception as e: print("FAILED:",repr(e))
        gc.collect();
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return results,rows,betas,ok


In [ ]:
from contextualized_topic_models.models.ctm import CombinedTM
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

def run_combinedtm(name,data,coh):
    labels=data['labels']; vocab=data['vocabulary']; sem=np.asarray(data['semantic'],dtype=np.float32); texts,dictionary,corpus=coh
    print("Preparing dense fixed BoW for CombinedTM...")
    bow_dense=data['bow'].toarray().astype(np.float32)
    id2token={int(i):str(w) for i,w in enumerate(vocab)}
    tp=TopicModelDataPreparation(); dataset=tp.load(sem,bow_dense,id2token)
    results=[]; rows=[]; betas=[]; ok=[]
    for seed in SEEDS:
        print("\n"+"-"*90); print(f"{name} | CombinedTM | seed={seed}"); print("-"*90); set_seed(seed); t=time.time()
        try:
            model=CombinedTM(bow_size=bow_dense.shape[1],contextual_size=sem.shape[1],n_components=K,model_type='prodLDA',hidden_sizes=CTM_HIDDEN,activation='softplus',dropout=DROPOUT,learn_priors=True,batch_size=BATCH_SIZE,lr=LEARNING_RATE,num_epochs=MAX_EPOCHS,num_data_loader_workers=0)
            model.fit(dataset,verbose=False,n_samples=CTM_N_SAMPLES_THETA,do_train_predictions=False)
            beta=model.get_topic_word_distribution(); theta=model.get_doc_topic_distribution(dataset,n_samples=CTM_N_SAMPLES_THETA)
            r,rr,beta=build_result(name,'CombinedTM',seed,beta,theta,labels,vocab,texts,dictionary,corpus,time.time()-t)
            results.append(r); rows.extend(rr); betas.append(beta); ok.append(seed); show_result(r); del model,theta,beta
        except Exception as e: print("FAILED:",repr(e))
        gc.collect();
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    del dataset,tp,bow_dense; gc.collect()
    return results,rows,betas,ok


In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

def bertopic_beta(model,assignments):
    ids=sorted(int(t) for t in np.unique(assignments) if int(t)!=-1)
    if not ids: raise RuntimeError('BERTopic produced no non-outlier topics.')
    ctfidf=model.c_tf_idf_; offset=int(getattr(model,'_outliers',0)); rows=[]
    for topic_id in ids:
        row_index=topic_id+offset
        if row_index<0 or row_index>=ctfidf.shape[0]: raise RuntimeError(f'Cannot map BERTopic topic {topic_id}.')
        rows.append(ctfidf[row_index].toarray().ravel().astype(np.float64))
    return normalize_beta(np.vstack(rows))

def run_bertopic(name,data,coh):
    docs=data['documents']['text_clean'].fillna('').astype(str).tolist(); emb=np.asarray(data['semantic'],dtype=np.float32); labels=data['labels']; vocab=data['vocabulary']; texts,dictionary,corpus=coh
    vmap={str(w):int(i) for i,w in enumerate(vocab)}
    results=[]; rows=[]; betas=[]; ok=[]
    for seed in SEEDS:
        print("\n"+"-"*90); print(f"{name} | BERTopic | seed={seed}"); print("-"*90); set_seed(seed); t=time.time()
        try:
            vectorizer=CountVectorizer(vocabulary=vmap,lowercase=False,token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9+#']+\b")
            umap_model=UMAP(n_neighbors=BERTOPIC_N_NEIGHBORS,n_components=BERTOPIC_UMAP_COMPONENTS,min_dist=0.0,metric='cosine',random_state=seed,low_memory=True)
            hdbscan_model=HDBSCAN(min_cluster_size=BERTOPIC_MIN_CLUSTER_SIZE,metric='euclidean',cluster_selection_method='eom',prediction_data=False)
            model=BERTopic(embedding_model=None,umap_model=umap_model,hdbscan_model=hdbscan_model,vectorizer_model=vectorizer,nr_topics=K,top_n_words=max(TOP_N,20),calculate_probabilities=False,verbose=False)
            assignments,_=model.fit_transform(docs,embeddings=emb); assignments=np.asarray(assignments,dtype=int); beta=bertopic_beta(model,assignments)
            r,rr,beta=build_result(name,'BERTopic',seed,beta,assignments,labels,vocab,texts,dictionary,corpus,time.time()-t,True)
            r['outlier_fraction']=float(np.mean(assignments==-1)); results.append(r); rows.extend(rr); betas.append(beta); ok.append(seed); show_result(r); print(f"Outlier fraction={r['outlier_fraction']:.4f}"); del model,assignments,beta
        except Exception as e: print("FAILED:",repr(e))
        gc.collect();
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return results,rows,betas,ok


In [ ]:
all_results=[]; all_topic_rows=[]; stability_rows=[]; pairwise_rows=[]

for name in DATASETS:
    print("\n"+"#"*90); print("DATASET:",name); print("#"*90)
    data=load_dataset(name); coh=coherence_inputs(data['documents'],data['vocabulary'])
    print("Documents:",data['bow'].shape[0],"| Vocabulary:",data['bow'].shape[1],"| Semantic:",data['semantic'].shape)
    for model_name,runner in [('ProdLDA',run_prodlda),('CombinedTM',run_combinedtm),('BERTopic',run_bertopic)]:
        print("\n"+"="*90); print(name,'—',model_name); print("="*90)
        results,rows,betas,ok=runner(name,data,coh); all_results.extend(results); all_topic_rows.extend(rows)
        if len(betas)>=2:
            st=multiseed_stability(betas,ok)
            stability_rows.append({'dataset':name,'model':model_name,'Mean_TS':st['mean'],'Std_TS':st['std'],'CI95_low':st['ci_low'],'CI95_high':st['ci_high'],'pairwise_comparisons':st['n_pairs'],'successful_runs':len(betas)})
            for rec in st['records']: pairwise_rows.append({'dataset':name,'model':model_name,**rec})
            print(f"Topic Stability: {st['mean']:.4f} ± {st['std']:.4f} (95% CI {st['ci_low']:.4f}–{st['ci_high']:.4f})")
        else:
            stability_rows.append({'dataset':name,'model':model_name,'Mean_TS':np.nan,'Std_TS':np.nan,'CI95_low':np.nan,'CI95_high':np.nan,'pairwise_comparisons':0,'successful_runs':len(betas)})
        del betas; gc.collect();
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    del data,coh; gc.collect()

runs_df=pd.DataFrame(all_results); topics_df=pd.DataFrame(all_topic_rows); stability_df=pd.DataFrame(stability_rows); pairwise_df=pd.DataFrame(pairwise_rows)
if runs_df.empty: raise RuntimeError('No Stage 08C-3 run completed successfully.')

metrics=['Cv','C_NPMI','UMass','Topic_Diversity','Topic_Redundancy','NMI','ARI']; summary=[]
for (dataset,model),g in runs_df.groupby(['dataset','model']):
    row={'dataset':dataset,'model':model,'runs':len(g),'mean_topics_observed':float(g['n_topics_observed'].mean())}
    for metric in metrics:
        vals=g[metric].astype(float).to_numpy(); mean=float(vals.mean()); std=float(vals.std(ddof=1)) if len(vals)>1 else 0.0; se=std/math.sqrt(len(vals))
        row[metric+'_mean']=mean; row[metric+'_std']=std; row[metric+'_CI95_low']=mean-1.96*se; row[metric+'_CI95_high']=mean+1.96*se
    sm=stability_df[(stability_df.dataset==dataset)&(stability_df.model==model)]
    if len(sm)==1:
        s=sm.iloc[0]; row['Topic_Stability_mean']=s.Mean_TS; row['Topic_Stability_std']=s.Std_TS; row['Topic_Stability_CI95_low']=s.CI95_low; row['Topic_Stability_CI95_high']=s.CI95_high
    if 'outlier_fraction' in g.columns:
        x=g['outlier_fraction'].dropna().to_numpy()
        if len(x): row['BERTopic_outlier_fraction_mean']=float(x.mean())
    summary.append(row)
summary_df=pd.DataFrame(summary)

article_table=pd.DataFrame({'Dataset':summary_df.dataset,'Model':summary_df.model,'Cv':summary_df.Cv_mean,'CNPMI':summary_df.C_NPMI_mean,'UMass':summary_df.UMass_mean,'TD':summary_df.Topic_Diversity_mean,'TR':summary_df.Topic_Redundancy_mean,'TS':summary_df.Topic_Stability_mean,'NMI':summary_df.NMI_mean,'ARI':summary_df.ARI_mean})
num=['Cv','CNPMI','UMass','TD','TR','TS','NMI','ARI']; article_table[num]=article_table[num].round(4)

runs_file=os.path.join(OUTPUT_DIR,'stage08c3_all_runs.csv'); summary_file=os.path.join(OUTPUT_DIR,'stage08c3_summary.csv'); stability_file=os.path.join(OUTPUT_DIR,'stage08c3_stability_summary.csv'); pairwise_file=os.path.join(OUTPUT_DIR,'stage08c3_pairwise_stability.csv'); topics_file=os.path.join(OUTPUT_DIR,'stage08c3_topic_words.csv'); article_file=os.path.join(OUTPUT_DIR,'stage08c3_article_table.csv')
runs_df.to_csv(runs_file,index=False); summary_df.to_csv(summary_file,index=False); stability_df.to_csv(stability_file,index=False); pairwise_df.to_csv(pairwise_file,index=False); topics_df.to_csv(topics_file,index=False); article_table.to_csv(article_file,index=False)

# Merge prior valid results
parts=[]
f1=os.path.join(STAGE08C1_DIR,'stage08c1_article_table.csv'); f2=os.path.join(STAGE08C2R_DIR,'stage08c2r_article_table.csv')
if os.path.exists(f1): parts.append(pd.read_csv(f1))
else: print('WARNING: Stage 08C-1 table not found.')
if os.path.exists(f2): parts.append(pd.read_csv(f2))
else: print('WARNING: Stage 08C-2R table not found.')
parts.append(article_table)
final=pd.concat(parts,ignore_index=True)
order={'LDA':1,'NMF':2,'ProdLDA':3,'CombinedTM':4,'BERTopic':5,'HMRNet-Final':6}; final['_order']=final.Model.map(order).fillna(99); final=final.sort_values(['Dataset','_order']).drop(columns='_order').reset_index(drop=True)
final_file=os.path.join(OUTPUT_DIR,'stage08c_FINAL_ALL_MODELS_article_table.csv'); final.to_csv(final_file,index=False)

config_out={'experiment':'Stage 08C-3','datasets':DATASETS,'seeds':SEEDS,'K':K,'top_n':TOP_N,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'max_epochs':MAX_EPOCHS,'prodlda_patience':PATIENCE,'prodlda_validation_fraction':VALIDATION_FRACTION,'dropout':DROPOUT,'combinedtm_hidden':list(CTM_HIDDEN),'combinedtm_theta_samples':CTM_N_SAMPLES_THETA,'bertopic_nr_topics':K,'bertopic_min_cluster_size':BERTOPIC_MIN_CLUSTER_SIZE,'bertopic_n_neighbors':BERTOPIC_N_NEIGHBORS,'bertopic_umap_components':BERTOPIC_UMAP_COMPONENTS,'semantic_embeddings':'Stage 08B 384-D semantic embeddings','package_versions':versions}
with open(os.path.join(OUTPUT_DIR,'stage08c3_experiment_config.json'),'w') as f: json.dump(config_out,f,indent=2)

print("\n"+"="*90); print('STAGE 08C-3 RESULTS'); print("="*90); display(article_table)
print("\n"+"="*90); print('FINAL ALL-MODEL CROSS-DATASET TABLE'); print("="*90); display(final)
print("\n"+"="*90); print('STAGE 08C-3 COMPLETED SUCCESSFULLY'); print("="*90)
print('1.',final_file); print('2.',article_file); print('3.',stability_file); print('4.',summary_file); print('5.',runs_file); print('6.',topics_file)
